- Acesse o site do Mistral e crie uma conta: https://mistral.ai/
- Configure um plano do tipo "Experimental"
- Gere uma chave API.
- Acesse a documentação do LangChain: https://python.langchain.com/docs/integrations/chat/mistralai/

In [2]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()
#from langchain.agents import initialize_agent, Tool
#from langchain.agents import AgentType

/workspaces/ml-supervised-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    print("API KEY não fornecida")

In [4]:
'''
Cria objeto para consumir modelos do Mistral. Outros provedores são bem parecidos.
Link da documentação: https://python.langchain.com/api_reference/mistralai/chat_models/langchain_mistralai.chat_models.ChatMistralAI.html#langchain_mistralai.chat_models.ChatMistralAI.get_num_tokens_from_messages
'''

llm = ChatMistralAI(
    model="open-mistral-7b",
    temperature=0,
    max_retries=1,
    verbose=True # Ajuda a ver tokens e modelo
)

In [5]:
'''
Forma mais simples de iniciar chat com modelos do Mistral
'''

mensagens = [
    (
        "system",
        "Você é um tradutor experiente que realiza traduções do português para o inglês. Traduza as sentenças enviadas.",
    ),
    ("human", "Hobbit não é uma medida de informação"),
]
ai_msg = llm.invoke(mensagens)
ai_msg

AIMessage(content='Hobbit is not a unit of measurement.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 54, 'total_tokens': 64, 'completion_tokens': 10}, 'model_name': 'open-mistral-7b', 'model': 'open-mistral-7b', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--9197ae7a-e9b2-4bdb-8d8f-1ddb86f198e3-0', usage_metadata={'input_tokens': 54, 'output_tokens': 10, 'total_tokens': 64})

Abaixo, podemos ver que é possível acessar propriedades importantes das nossas chamadas a API do provedor

In [6]:
ai_msg.content

'Hobbit is not a unit of measurement.'

In [7]:
ai_msg.response_metadata['token_usage']

{'prompt_tokens': 54, 'total_tokens': 64, 'completion_tokens': 10}

In [8]:
ai_msg.usage_metadata

{'input_tokens': 54, 'output_tokens': 10, 'total_tokens': 64}

In [9]:
llm.get_num_tokens("Quantos tokens possui essa mensagem?")

12

In [10]:
'''
O ChatPromptTemplate permite a formatação dinâmica de nossos prompts. Muito comum
quando temos muitos prompts para vários cenários e esses são organizados em arquivos
separados, como por exemplo no formato .jinja2
'''

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

'''
Existem tipos de dados do Langchain (runnables) que podem ser conectados via operador
pip "|". Isso faz com que o output de um seja enviado como inputo de outro.
'''
chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

AIMessage(content='Ich liebe das Programmieren.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 21, 'total_tokens': 30, 'completion_tokens': 9}, 'model_name': 'open-mistral-7b', 'model': 'open-mistral-7b', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--e2852eac-1612-4b15-81d7-b806b1af6851-0', usage_metadata={'input_tokens': 21, 'output_tokens': 9, 'total_tokens': 30})